In [2]:
# 🚀 Import necessary libraries
import jax
import jax.numpy as jnp
import jax.random as random
from jax import grad, jit, vmap, lax
import optax  # Optimizer library

# ✅ Define Recursion Constraints
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8

# ✅ Define Dynamic Pi & Phi Functions With Constraints
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

# ✅ Update DPPU Processing With JAX-Compatible Recursion Safeguards
def dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=1.0):
    depth = jnp.minimum(jnp.array(depth, dtype=jnp.int32), MAX_RECURSION_DEPTH)

    @jit
    def compute(x):
        prev_x = x

        def body_fn(i, x):
            pi_dynamic = dynamic_pi(i, scale_factor)
            phi_dynamic = dynamic_phi(i, scale_factor)

            scaling = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
            new_x = jnp.sin(x * scaling * pi_dynamic) * jnp.exp(-x / (phi_dynamic + 1))

            divergence = jnp.abs(new_x - prev_x).sum()
            stop_condition = divergence > 1e3

            x = lax.cond(
                stop_condition,
                lambda _: prev_x,
                lambda _: new_x,
                operand=None
            )
            return x

        x = lax.fori_loop(0, depth, body_fn, x)
        return x

    return compute(x)

# ✅ Vectorized Processing for Batch Computation
batch_size = 100
data_size = 1000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jnp.tile(batch_input, (batch_size, 1))

batched_dppu_processing = vmap(lambda x: dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=0.5), in_axes=0)
output_batch = batched_dppu_processing(batch_input)

print("Batch Output Shape:", output_batch.shape)

import time
import jax
import jax.numpy as jnp

# ✅ Ensure this function is already defined in your script
# Remove the incorrect import statement

# ✅ Benchmarking Configuration
NUM_TRIALS = 10
INPUT_SIZE = 1000

# ✅ Warm-up (JIT compile)
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)))

# ✅ Run multiple trials & measure execution time
times = []
for _ in range(NUM_TRIALS):
    start = time.time()
    result = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)))
    _ = jax.device_get(result)
    end = time.time()
    times.append(end - start)

# ✅ Compute & Display Benchmark Results
avg_time = sum(times) / len(times)
min_time = min(times)
max_time = max(times)

print(f"\n🔥 Benchmark Results (Input Size: {INPUT_SIZE}, Trials: {NUM_TRIALS}) 🔥")
print(f"✅ Average Execution Time: {avg_time:.6f} seconds")
print(f"✅ Fastest Execution Time: {min_time:.6f} seconds")
print(f"✅ Slowest Execution Time: {max_time:.6f} seconds")

jax.devices()




Batch Output Shape: (100, 1000)

🔥 Benchmark Results (Input Size: 1000, Trials: 10) 🔥
✅ Average Execution Time: 0.278528 seconds
✅ Fastest Execution Time: 0.161368 seconds
✅ Slowest Execution Time: 0.455646 seconds


[CpuDevice(id=0)]